In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Load Stock_TCJM ───────────────────────────────────────────────────────────
f         = np.load('Stock_TCJM.npz', allow_pickle=True)
Stock_TCJM = f['data']           # (501, 501, 4, 7): [time, cohort, type, material]
types      = list(f['types'])    # dwelling type names (axis 2)
materials  = list(f['materials'])# material names (axis 3)
years      = f['years']          # 1600–2100

print("Stock_TCJM shape:", Stock_TCJM.shape)
print("Types:", types)
print("Materials:", materials)

In [ ]:
# ── Carbon intensity (GWP) per material [kgCO2e / kg] ────────────────────────
# From EPD data collected in ghg epds.xlsx
ghg_df = pd.read_excel("ghg epds.xlsx", skiprows=5)
CI = ghg_df[["Material", "GWP avg"]].head(7)
print(CI.to_string(index=False))

# GWP vector — order must match materials axis in Stock_TCJM
# Stock_TCJM materials order: concrete, brick, wood, steel, glass, aluminum, copper
# GWP file order:             Concrete, Brick, Wood, Steel, Glass, Aluminium, Copper  ✓
gwp = CI["GWP avg"].values  # (7,) [kgCO2e/kg]

In [ ]:
# ── Emissions: embodied carbon at time of construction ────────────────────────
# Emissions happen when materials are manufactured, i.e. when the building is built.
# For cohort c, this occurs at time t = c (the construction year).
# So: Inflow_CJM[c, j, m] = Stock_TCJM[c, c, j, m]  ← diagonal of the time-cohort axes
#
# This extracts all 501 diagonal entries at once (no loop needed):
n_c = Stock_TCJM.shape[1]
Inflow_CJM = Stock_TCJM[np.arange(n_c), np.arange(n_c), :, :]  # (501, 4, 7): [cohort, type, material]

# Emissions_CJM[c, j, m] = Inflow_CJM[c, j, m] * GWP[m]
# = kgCO2e emitted when cohort c, type j, material m is built
Emissions_CJM = np.einsum('cjm,m->cjm', Inflow_CJM, gwp)  # (501, 4, 7)

print("Emissions_CJM shape:", Emissions_CJM.shape)
print("Dimensions: [cohort, type, material]")

In [ ]:
# ── Validation ────────────────────────────────────────────────────────────────
# 1. Spot-check: manually compute emissions for one cohort/type cell
c_idx = np.argmin(np.abs(years - 1990))  # cohort built in 1990
j_idx = types.index('Parcelhus')
manual     = np.dot(Inflow_CJM[c_idx, j_idx, :], gwp)
einsum_val = Emissions_CJM[c_idx, j_idx, :].sum()
print(f"Spot-check (cohort 1990, Parcelhus): manual = {manual:.1f}, einsum = {einsum_val:.1f}, match = {np.isclose(manual, einsum_val)}")

# 2. Total emissions by construction year (sum over type and material)
emissions_by_year = Emissions_CJM.sum(axis=(1, 2))  # (501,): kgCO2e per year

idx_1960 = np.argmin(np.abs(years - 1960))
idx_2020 = np.argmin(np.abs(years - 2020))
print(f"\nEmbodied carbon emitted per construction year [Mt CO2e]:")
for yr in [1960, 1970, 1980, 1990, 2000, 2010, 2020]:
    idx = np.argmin(np.abs(years - yr))
    print(f"  {yr}: {emissions_by_year[idx]/1e9:.2f} Mt CO2e")

# 3. Total cumulative emissions 1600–2020
cumulative = Emissions_CJM[:idx_2020+1].sum()
print(f"\nCumulative embodied carbon 1600–2020: {cumulative/1e9:.1f} Mt CO2e")

# 4. Breakdown by material for all cohorts
print("\nTotal emissions by material [Mt CO2e] (all cohorts 1600–2100):")
for m, mat in enumerate(materials):
    val = Emissions_CJM[:, :, m].sum()
    print(f"  {mat:10s}: {val/1e9:.1f}")

In [ ]:
# ── Cumulative emissions over time ───────────────────────────────────────────
# Cumulative sum along the cohort (time) axis gives total emissions up to each year
emissions_by_type = Emissions_CJM.sum(axis=2)              # (501, 4): sum over materials
cumulative_by_type = np.cumsum(emissions_by_type, axis=0)  # (501, 4): running total

fig, ax = plt.subplots(figsize=(10, 5))
ax.stackplot(years, [cumulative_by_type[:, j] / 1e9 for j in range(len(types))],
             labels=types)
ax.set_xlabel("Year")
ax.set_ylabel("Cumulative embodied carbon [Mt CO₂e]")
ax.set_title("Cumulative embodied carbon from dwelling construction")
ax.set_xlim(1600, 2100)
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# ── Cumulative emissions — total across all types ────────────────────────────
cumulative_total = cumulative_by_type.sum(axis=1)  # (501,): sum over types

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(years, cumulative_total / 1e9)
ax.set_xlabel("Year")
ax.set_ylabel("Cumulative embodied carbon [Mt CO₂e]")
ax.set_title("Cumulative embodied carbon from dwelling construction (all types)")
ax.set_xlim(1600, 2100)
plt.tight_layout()
plt.show()

In [ ]:
# ── Annual embodied carbon — total across all types ──────────────────────────
# Each point = embodied carbon emitted that year from new construction
emissions_total = emissions_by_type.sum(axis=1)  # (501,): sum over types

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(years, emissions_total / 1e9)
ax.set_xlabel("Year")
ax.set_ylabel("Embodied carbon [Mt CO₂e / year]")
ax.set_title("Annual embodied carbon from new dwelling construction (all types)")
ax.set_xlim(1600, 2100)
ax.set_ylim(0, 6)
plt.tight_layout()
plt.show()